# Numerical IK Math

## 1) State, target, and units

### Joint vector
$$
q = [q_1,q_2,q_3,q_4,q_5,q_6]^\top \in \mathbb{R}^6
$$

IK is solved in **radians**.

### Cartesian pose target format
Each target is a 6-vector:
$$
x_{\text{target}} = [p_x,p_y,p_z,r_x,r_y,r_z]^\top
$$

- first 3 entries: desired position
- last 3 entries: **rotation vector** (axis-angle packed as a vector)

So orientation is **not Euler angles**.

---

## 2) Homogeneous transforms (used in `kinematics.py`)

A rigid transform is built as

$$
T(R,t)=
\begin{bmatrix}
R & t\\
0 & 1
\end{bmatrix}
$$

where $R\in\mathbb{R}^{3\times3}$ and $t\in\mathbb{R}^3$.

The end-effector transform is produced by the explicit transform chain in `forwardKinematicsT(q)`:

$$
T_{\text{EE}}(q)=
T_0\,
T_1(q_1)\,
T_2(q_2)\,
T_3(q_3)\,
T_4(q_4)\,
T_5(q_5)\,
T_6(q_6)
$$

From the final transform,

$$
T_{\text{EE}}(q)=
\begin{bmatrix}
R_{\text{cur}} & p_{\text{cur}}\\
0 & 1
\end{bmatrix}
$$

we extract:
- current position $p_{\text{cur}}$
- current rotation matrix $R_{\text{cur}}$

---

## 3) Rotation vector math (`vectorToR`, `rToVector`)

A rotation vector $r\in\mathbb{R}^3$ stores axis-angle as:

$$
r = \theta \hat{k}
$$

where:
- $\theta = \|r\|$ is rotation angle
- $\hat{k}$ is the unit axis

### Rotation vector $\to$ rotation matrix (Rodrigues)
If $K=[\hat{k}]_\times$, then:

$$
R = I + \sin\theta\,K + (1-\cos\theta)K^2
$$

This is implemented in `vectorToR()`.

### Rotation matrix $\to$ rotation vector
Given a rotation matrix $R$, the angle is

$$
\theta = \cos^{-1}\!\left(\frac{\operatorname{tr}(R)-1}{2}\right)
$$

Then:
- if $\theta \approx 0$, the solver returns the zero vector
- if $\theta$ is not near $\pi$, it uses the standard skew-symmetric formula
- if $\theta \approx \pi$, it uses a more robust diagonal-based fallback

This is implemented in `rToVector()`.

---

## 4) Pose error used by IK (`poseError`)

From FK, current pose is:
$$
T(q)=\begin{bmatrix}R_{\text{cur}} & p_{\text{cur}}\\0&1\end{bmatrix}
$$

Given target $x_{\text{target}}=[p_{\text{des}}, r_{\text{des}}]$:
- $R_{\text{des}}=\text{vectorToR}(r_{\text{des}})$

### Position error
$$
e_{\text{pos}} = p_{\text{des}} - p_{\text{cur}}
$$

### Orientation error
$$
R_{\text{err}} = R_{\text{des}}R_{\text{cur}}^\top
$$
$$
e_{\text{rot}} = \text{rToVector}(R_{\text{err}})
$$

### Weighted 6D residual
$$
e(q)=
\begin{bmatrix}
w_{\text{pos}}\,e_{\text{pos}}\\
w_{\text{rot}}\,e_{\text{rot}}
\end{bmatrix}
\in \mathbb{R}^6
$$

Typical current defaults in the code are:
- $w_{\text{pos}}=1.0$
- $w_{\text{rot}}=0.001$

So orientation error is intentionally weighted much lower than position error.

---

## 5) Jacobian by central finite differences (`jacobianFd`)

The Jacobian is **numerical** (not analytic):

$$
J(q)=\frac{\partial e}{\partial q}\in\mathbb{R}^{6\times 6}
$$

Each column is approximated by **central finite differences**:

$$
J_{:,i}\approx\frac{e(q+h_i e_i)-e(q-h_i e_i)}{2h_i}
$$

where:
- $e_i$ is the $i$-th basis vector
- $h_i$ is the finite-difference step for joint $i$

The code uses an adaptive per-joint step:
$$
h_i = h\max(1,|q_i|)
$$

with default base step:
$$
h = 10^{-5}
$$

This is more accurate than the older forward-difference approximation.

---

## 6) SVD-based damped least-squares IK step (`dampedSvdStep`)

At each iteration, compute:
- residual $e$
- numerical Jacobian $J$

Then take the singular value decomposition:
$$
J = U\Sigma V^\top
$$

where
$$
\Sigma = \operatorname{diag}(\sigma_1,\sigma_2,\dots,\sigma_6)
$$

The damped task-space step is computed as

$$
\Delta q_{\text{task}}
=
- V
\operatorname{diag}\left(
\frac{\sigma_i}{\sigma_i^2+\lambda^2}
\right)
U^\top e
$$

This is the SVD form of damped least squares and is more numerically stable near singularities than directly solving normal equations.

---

## 7) Continuity / smoothing term

To encourage continuity with the previous solution $q_{\text{prev}}$, the solver adds:

$$
\Delta q_{\text{smooth}} = -\mu(q-q_{\text{prev}})
$$

where:
- $\mu$ = `smoothw`

The full proposed step is

$$
\Delta q = \Delta q_{\text{task}} + \Delta q_{\text{smooth}}
$$

This discourages branch jumps and large posture changes between neighboring waypoints.

---

## 8) Adaptive damping near singularities (`adaptiveDamping`)

The code uses the smallest singular value

$$
\sigma_{\min} = \min_i \sigma_i
$$

to increase damping near singularities.

Let:
- $\lambda_0$ = base damping
- $\sigma_{\text{th}}$ = singularity threshold
- $g$ = singularity gain

Then

$$
\lambda =
\begin{cases}
\lambda_0 + g\left(1-\dfrac{\sigma_{\min}}{\sigma_{\text{th}}}\right)^2,
& \sigma_{\min}<\sigma_{\text{th}}\\[8pt]
\lambda_0,
& \sigma_{\min}\ge \sigma_{\text{th}}
\end{cases}
$$

So:
- away from singularities, damping stays small
- near singularities, damping increases smoothly

---

## 9) Step scaling and maximum joint step

After computing the step, the code applies an overall scaling factor:

$$
\Delta q \leftarrow \alpha \Delta q
$$

where:
- $\alpha$ = `stepScale`

Then the step norm is limited to a maximum value:

$$
\|\Delta q\| \le \Delta q_{\max}
$$

If the proposed step is too large, it is scaled down.

This makes the solver more conservative and helps avoid large jumps.

---

## 10) Trust-region-like step acceptance (`solvePoseIk`)

The solver does **not** automatically accept the first proposed step.

For each iteration it:
1. computes a candidate step
2. applies it
3. evaluates the new residual norm

A step is accepted only if it reduces the error:

$$
\|e(q+\Delta q)\| < \|e(q)\|
$$

If the step is not good enough, the solver:
- rejects it
- increases damping
- retries

So the current solver behaves like a simple trust-region / retry strategy:
- try a step
- if it helps, keep it
- if not, become more conservative

---

## 11) Stopping conditions

The solver stops when one of the following happens:

### Converged
If the weighted residual norm is small enough:
$$
\|e(q)\| < \text{tol}
$$

### Very small accepted step
If the accepted step norm becomes extremely small, the solver treats that as stagnation.

### Very small improvement
If the error decrease remains too small over several iterations, the solver stops.

### No improving step found
If repeated retries with larger damping still do not reduce the error, the solver stops.

### Maximum iterations reached
If none of the earlier conditions occur, the solver stops after `maxIters`.

---

## 12) Joint limits and clamping (`clampQ`)

For lower and upper bounds $q^{\min}, q^{\max}$, clamping is:

$$
q_{\text{cmd}}=\min(\max(q,q^{\min}),q^{\max})
$$

The clamp function also reports:
- whether any limit was hit
- which joint indices violated limits

---

## 13) Reach checks (`checkReach`)

Before IK, the target position is pre-filtered using simple checks:
- radial max: $\|p\|\le r_{\max}$
- radial min: $\|p\|\ge r_{\min}$
- per-coordinate exclusion zone: $|x|,|y|,|z|\ge r_{\min\_reach}$

These are quick sanity checks, not a full exact workspace test.

---

## 14) Trajectory generation (`generateTrajectoryPose`)

Given pose targets $x_0,\dots,x_{N-1}$, IK is solved **sequentially**.

For each target:
1. start from the previous solution $q_{\text{prev}}$
2. solve IK
3. decide whether that solution is good enough
4. if needed, try a few nearby fallback seeds
5. score the candidates and keep the best one

So the trajectory solver is not just warm-started IK. It is:

- warm-started sequential IK
- with continuity bias
- with limited nearby fallback seeds

---

## 15) Candidate scoring during fallback seed search

If fallback seeds are used, candidate solutions are scored using:
- pose error
- continuity relative to the previous solution

A typical score has the form

$$
\text{score}
=
\|e(q)\| + c\|q-q_{\text{prev}}\|
$$

where $c$ is a small continuity weight.

The best candidate is chosen for that waypoint.

---

## 16) Target interpolation (`createJson.py` / `main.py`)

Targets are built by linear interpolation in pose-vector space:

$$
\text{targets}=\text{linspace}(\text{startPose},\text{goalPose},N)
$$

where each target is:

$$
[x,y,z,r_x,r_y,r_z]
$$

So interpolation happens directly in:
- Cartesian position
- rotation-vector coordinates

---

## 17) Singularity metric (`singularityCost`)

The code includes a singularity diagnostic based on the smallest singular value of the **weighted numerical Jacobian**.

If

$$
\sigma_{\min} = \min_i \sigma_i
$$

then the singularity cost is

$$
\text{cost}
=
\frac{1}{\sigma_{\min}^2+\varepsilon}
$$

A larger value means the Jacobian is closer to singular or ill-conditioned.

Important: because the Jacobian is built from the **weighted residual**, this is a singularity measure of the **weighted IK task**, not a pure geometric manipulability metric.

---

## 18) Trajectory retiming (`retimeTrajectoryLimits`)

After IK generates a joint trajectory, the timestamps are adjusted so that the motion respects:
- joint velocity limits
- TCP linear velocity limits
- joint acceleration limits

If the required minimum duration is greater than the allowed maximum duration,

$$
T_{\min} > T_{\max}
$$

then the code raises an error.

So retiming ensures that the final exported motion is dynamically feasible under the chosen limits.

---

## 19) JSON export

After retiming, the joint trajectory is exported in **degrees**:

$$
q_{\deg}=\operatorname{rad2deg}(q)
$$

Each waypoint in the JSON file contains:
- time stamp `t`
- joint vector `q`

So the exported file has the form:

```json
{
  "units": "deg",
  "waypoints": [
    {"t": ..., "q": [...]},
    ...
  ]
}